# Overshoot multi-scénarios

Figure "Overshoot / Safe Operating Space" en valeurs absolues : position de
chaque scénario (base-year + scénarios de transition 2050) par rapport à
chaque limite planétaire (LP), en multiples de la limite basse (1L = budget
France).

Le budget France est le budget mondial de `seuils.xlsx` (feuille "Synthèse")
multiplié par la part de la France lue dans `budget_shares.xlsx` (feuille
"Parts_regions_EXIOBASE", colonne "Part EPC ref.", ligne "FR") — principe de
partage `sharing_principle="EPC"`. Rien n'est divisé par la population.

Génère `figures/overshoot_safe_operating_space_all_scenarios.png`.

Variante par habitant (comparaison CBA/PBA) : voir `Figure comparaison.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

notebooks_dir = Path.cwd()
while not (notebooks_dir / "planefr_lib").exists() and notebooks_dir != notebooks_dir.parent:
    notebooks_dir = notebooks_dir.parent
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

import matplotlib.pyplot as plt

from planefr_lib import config, io, processing
from planefr_lib.plot_overshoot import create_overshoot_safe_space_figure_by_region

## 1. Chargement des données et traitement des scénarios

In [ ]:
facteurs_carac_df = io.load_facteurs_carac()
bridge_matrices_df = io.load_bridge_matrices()
seuils_df = io.load_seuils()
shares_df = io.load_budget_shares_df()

scenario_folders = config.get_scenario_folders(exclude_2019=True)
scenario_names = [f.name for f in scenario_folders]

all_scenarios_data = []
for scenario_folder in scenario_folders:
    print(f"Traitement : {scenario_folder.name}")
    data_by_subprocess, _ = processing.process_scenario(
        scenario_folder, facteurs_carac_df, bridge_matrices_df, seuils_df
    )
    all_scenarios_data.append(data_by_subprocess)

print(f"\n{len(all_scenarios_data)} scénarios traités.")

# Tous ces scénarios sont des scénarios France : leur nom de dossier ne contient
# aucun code EXIOBASE, ils sont donc rattachés à la ligne "FR" de budget_shares.xlsx
# (config.DEFAULT_REGION_CODE).
print("Parts du budget mondial utilisées :",
      {n: processing.resolve_region_code(n, shares_df) for n in scenario_names})

## 2. Figure overshoot (valeurs absolues)

In [ ]:
fig = create_overshoot_safe_space_figure_by_region(
    all_scenarios_data=all_scenarios_data,
    seuils_df=seuils_df,
    shares_df=shares_df,
    scenario_names=scenario_names,
    unit_row=config.UNIT_ROW_ABS,
    x_main_max=7.0,
    title="Overshoot relative to the Safe Operating Space for each PB and scenario",
    sharing_principle="EPC",
)

config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
output_path = config.FIGURES_DIR / "overshoot_safe_operating_space_all_scenarios.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Figure sauvegardée : {output_path}")
plt.show()